# 1. Análise Exploratória de Dados (EDA) e Limpeza



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de visualização
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

## 1.1 Carregamento dos Dados

In [ ]:
# Carregando o dataset de treino
df_train = pd.read_csv('train.csv')
print(f"O dataset possui {df_train.shape[0]} linhas e {df_train.shape[1]} colunas.\n")
df_train.head()

## 1.2 Limpeza de Dados (Tratamento de Valores Nulos)
Vamos analisar os valores ausentes e tratá-los de acordo com o dicionário de dados (`data_description.txt`). Muitos "nulos" na verdade significam a ausência da característica (ex: a casa não tem piscina).

In [ ]:
# Verificando valores nulos antes da limpeza
missing = df_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Valores Nulos ANTES da limpeza:")
print(missing)

# Tratamento baseado no data_description.txt
# Colunas onde NA significa "Ausência da característica" (Não é um dado faltante real, a casa apenas não possui o item)
cols_na_is_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
                   'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                   'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                   'MasVnrType']

for col in cols_na_is_none:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna('None')

# Para variáveis numéricas associadas a essas características ausentes (ex: área da garagem), preencheremos com 0
cols_na_is_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 
                   'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 
                   'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_na_is_zero:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(0)

# Para a LotFrontage (Comprimento da rua conectada à propriedade), podemos preencher com a mediana do bairro (Neighborhood)
df_train['LotFrontage'] = df_train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Para o restante das variáveis categóricas com poucos valores faltantes (ex: Electrical), preenchemos com o valor mais comum (moda)
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])

print("\nVerificando se ainda há valores nulos após a limpeza:")
print("Total de nulos restantes:", df_train.isnull().sum().max())

## 1.3 Distribuição da Variável Alvo (`SalePrice`)
O objetivo principal do desafio é prever o preço de venda das casas, portanto é importante analisarmos essa variável.

In [ ]:
# Plotando a distribuição do Preço de Venda
plt.figure(figsize=(10, 6))
sns.histplot(df_train['SalePrice'], kde=True, bins=50, color='blue')
plt.title('Distribuição de SalePrice (Preço de Venda)', fontsize=14)
plt.xlabel('SalePrice ($)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.show()

print(f"Média do Preço: ${df_train['SalePrice'].mean():.2f}")
print(f"Mediana do Preço: ${df_train['SalePrice'].median():.2f}")
print("Assimetria (Skewness): %f" % df_train['SalePrice'].skew())
print("Curtose (Kurtosis): %f" % df_train['SalePrice'].kurt())

**Observação:** A distribuição tem assimetria positiva (cauda longa à direita). Normalmente, em modelos de regressão linear, pode ser útil aplicar uma transformação logarítmica em variáveis assimétricas na fase de Feature Engineering.

## 1.4 Matriz de Correlação
Identificando as correlações entre as variáveis numéricas para saber quais características mais afetam o preço da casa.

In [ ]:
# Selecionar apenas colunas numéricas para calcular correlação
numeric_cols = df_train.select_dtypes(include=[np.number])

# Calcular a matriz de correlação
corr_matrix = numeric_cols.corr()

# Extrair as variáveis com maior correlação (positiva ou negativa) com SalePrice
top_corr = corr_matrix['SalePrice'].sort_values(ascending=False).head(15)
print("Top 15 Variáveis Mais Correlacionadas com SalePrice:\n")
print(top_corr)

# Plotar o Heatmap das variáveis mais correlacionadas
plt.figure(figsize=(12, 10))
top_corr_cols = top_corr.index
sns.heatmap(df_train[top_corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", square=True, linewidths=.5)
plt.title('Matriz de Correlação (Top 15 Atributos vs SalePrice)', fontsize=15)
plt.show()

## 1.5 Análise Bivariada (Variáveis vs Preço)
Abaixo vemos a relação das duas variáveis que têm a correlação mais forte com o preço: `OverallQual` (Qualidade Geral) e `GrLivArea` (Área útil acima do solo).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot de OverallQual vs SalePrice
sns.boxplot(x='OverallQual', y='SalePrice', data=df_train, ax=ax[0])
ax[0].set_title('SalePrice vs OverallQual (Qualidade Geral do Material e Acabamento)')
ax[0].set_xlabel('OverallQual (1 a 10)')
ax[0].set_ylabel('SalePrice ($)')

# Scatterplot de GrLivArea vs SalePrice
sns.scatterplot(x='GrLivArea', y='SalePrice', data=df_train, alpha=0.6, ax=ax[1], color='darkred')
ax[1].set_title('SalePrice vs GrLivArea (Área Habitável Acima do Nível do Solo)')
ax[1].set_xlabel('GrLivArea (Pés Quadrados)')
ax[1].set_ylabel('SalePrice ($)')

plt.tight_layout()
plt.show()

Nota-se um claro aumento exponencial e linear respectivamente nessas duas variáveis. 
No gráfico de dispersão (`GrLivArea`), é possível observar também a presença de dois fortes **outliers** no canto inferior direito (casas muito grandes, mas vendidas por preços baixos). No passo de Feature Engineering eles poderiam ser removidos do dataset de treino.